In [1]:
from utils.logger import logger
from utils.config import config
from pipeline.pdf_parser import GrobidPDFParser
from pipeline.sentence_extractor import extract_sentences
from pipeline.reference_resolver import ReferenceResolver
from pipeline.classifier import CitationClassifier
from pipeline.contribution_profile import ContributionProfileExtractor, build_classifier_system_prompt
from pipeline.retriever import HybridRetriever
from pipeline.urgency_scorer import UrgencyScorer
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from pipeline.claim_decomposer import ClaimDecomposer
from entities.decomposition import Decomposition
from utils.citation_remover import remove_random_citations
from database.qdrant.store import create_qdrant_client

c:\Users\sampe\OneDrive\Desktop\facultate\licenta\missing-citations-identifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logger.info("Starting the application...")

logger.info("Parsing the PDF and extracting information...")

parser = GrobidPDFParser(pdf_path="../papers/BERT.pdf")
parsed_paper = parser.parse()

logger.info("Successfully parsed the PDF. Extracted information:")
logger.info(f"Title: {parsed_paper.title}")
logger.info(f"Abstract: {parsed_paper.abstract}")

2026-05-10 21:16:43,035 - missing_citations - INFO - Starting the application...
2026-05-10 21:16:43,036 - missing_citations - INFO - Parsing the PDF and extracting information...
2026-05-10 21:16:46,395 - missing_citations - INFO - Successfully parsed the PDF. Extracted information:
2026-05-10 21:16:46,396 - missing_citations - INFO - Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
2026-05-10 21:16:46,397 - missing_citations - INFO - Abstract: We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a;[CITE:b36], BERT is designed to pretrain deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be finetuned with just one additional output layer to create state-of-the-art models for a wide ra

In [3]:
logger.info("Extracting sentences from the parsed paper...")
sentences = extract_sentences(parsed_paper)
logger.info(f"Extracted {len(sentences)} sentences from the paper.")


2026-05-10 21:16:46,407 - missing_citations - INFO - Extracting sentences from the parsed paper...
2026-05-10 21:16:49,459 - missing_citations - INFO - Extracted 287 sentences from the paper.


In [4]:
logger.info("Removing random citations from the sentences for testing...")
result = remove_random_citations(sentences, fraction=0.1, seed=23)

sentences = result.records
removed_indices = result.removed_indices
logger.info(f"Removed {len(removed_indices)} sentences containing citations for testing.")

logger.info(f"removed_indices: {removed_indices}")


2026-05-10 21:16:49,470 - missing_citations - INFO - Removing random citations from the sentences for testing...
2026-05-10 21:16:49,472 - missing_citations - INFO - Removed 6 sentences containing citations for testing.
2026-05-10 21:16:49,473 - missing_citations - INFO - removed_indices: [5, 11, 42, 44, 147, 266]


In [5]:
logger.info("Resolving references in the paper...")
resolver = ReferenceResolver()
resolved_references = []

for ref in parsed_paper.references:
    resolved = resolver.resolve(ref)
    resolved_references.append(resolved)

logger.info("Resolved references:")
for ref, resolved in zip(parsed_paper.references, resolved_references):
    logger.info(f"Original: {ref}")
    logger.info(f"Resolved: {resolved}")
    print("---")

logger.info(f"Stats: {resolver.stats}")


2026-05-10 21:16:49,486 - missing_citations - INFO - Resolving references in the paper...
2026-05-10 21:17:04,950 - missing_citations - INFO - Resolved references:
2026-05-10 21:17:04,951 - missing_citations - INFO - Original: Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.
2026-05-10 21:17:04,951 - missing_citations - INFO - Resolved: ResolvedReference(raw_reference='Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.', resolved_paper_id='W2880875857', openalex_id=None, title='Contextual String Embeddings for Sequence Labeling', doi=None, method='fuzzy_title', confidence=1.0, unresolved_reason=None)
---
2026-05-10 21:17:04,952 - missing_citations - INFO - Original: Rami Al-Rfou, D

In [6]:
if resolver.stats.get("openalex_external", -1) not in [-1, 0]:
    from sentence_transformers import SentenceTransformer
    from fastembed import SparseTextEmbedding
    from database.qdrant import create_qdrant_client
    from utils.config import config

    logger.info("Initializing models and Qdrant client...")

    # Initialize Qdrant client
    qdrant_client = create_qdrant_client(config.QDRANT_URL)
    logger.info(f"Connected to Qdrant at {config.QDRANT_URL}")

    # Load dense model
    logger.info(f"Loading dense model: {config.DENSE_MODEL}...")
    dense_model = SentenceTransformer(config.DENSE_MODEL)
    logger.info("Dense model loaded")

    # Load sparse model
    logger.info(f"Loading sparse model: {config.SPARSE_MODEL}...")
    sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)
    logger.info("Sparse model loaded")

In [7]:
from ingest_missing import ingest_openalex_papers
from indexer import EmbeddingIndex

# 1. Collect the IDs of all papers that were found externally
missing_ids = [
    ref.openalex_id 
    for ref in resolved_references 
    if ref.method == "openalex_external" and ref.openalex_id
]

if missing_ids:
    # 2. You will need to pass your initialized EmbeddingIndex. 
    # (Assuming you already have your qdrant_client, dense_model, etc. initialized)
    embedding_idx = EmbeddingIndex(
        qdrant_client=qdrant_client,
        dense_model=dense_model,
        sparse_model=sparse_model
    )
    
    # 3. Fetch, insert to Postgres, embed, and insert to Qdrant!
    inserted_count = ingest_openalex_papers(missing_ids, embedding_idx)
    print(f"Successfully ingested {inserted_count} missing papers into the local corpus.")


In [8]:
"""
STAGE 1.5 - Contribution Profile Extraction

Extracts a structured profile of what THIS paper proposes (novel) versus what it
uses (prior work). Injected into the classifier's system prompt so descriptive
sentences about the authors' novel contributions are not flagged as missing
citations.
"""

logger.info("Extracting paper contribution profile...")
profile_extractor = ContributionProfileExtractor(model=config.CLASSIFIER_BACKUP[1])
contribution_profile = profile_extractor.extract(parsed_paper)

logger.info(f"Contribution profile: {contribution_profile}")
logger.info(f"  proposes:  {list(contribution_profile.proposes)}")
logger.info(f"  uses:      {list(contribution_profile.uses)}")
logger.info(f"  novel:     {list(contribution_profile.novel_contributions)}")


2026-05-10 21:17:05,082 - missing_citations - INFO - Extracting paper contribution profile...
2026-05-10 21:17:06,212 - missing_citations - INFO - Extracting contribution profile from 2 section(s): ['Introduction', 'Conclusion']
2026-05-10 21:17:11,121 - missing_citations - INFO - Contribution profile: ContributionProfile(system_names=['BERT'], proposes=['BERT', 'Masked Language Model', 'Next Sentence Prediction', 'bidirectional Transformer pre-training'], uses=['Transformer', 'WordPiece embeddings', 'GELU activation', 'Cloze task', 'Adam optimizer', 'OpenAI GPT'], novel_contributions=4 bullets, source_sections=['Introduction', 'Conclusion'])
2026-05-10 21:17:11,122 - missing_citations - INFO -   proposes:  ['BERT', 'Masked Language Model', 'Next Sentence Prediction', 'bidirectional Transformer pre-training']
2026-05-10 21:17:11,122 - missing_citations - INFO -   uses:      ['Transformer', 'WordPiece embeddings', 'GELU activation', 'Cloze task', 'Adam optimizer', 'OpenAI GPT']
2026-05-

In [ ]:
"""
STAGE 4A - Citation Worthiness Classification with GEMINI CLASSIFIER
"""

logger.info("Classifying sentences for citation worthiness using Gemini Classifier...")

# Build a paper-specific system prompt that injects the contribution profile so
# the classifier knows which entities are this paper's novel work and which are
# external building blocks.
augmented_system_prompt = build_classifier_system_prompt(contribution_profile)

citation_classifier = CitationClassifier(
    model=config.CLASSIFIER_BACKUP[1],
    batch_size=31,
    delay_between_calls_seconds=50,
    system_prompt=augmented_system_prompt,
)

classified_sentences = citation_classifier.classify_sentences(sentences[:30], parsed_paper.title, parsed_paper.abstract)

logger.info("Classification results:")
for i, sentence in enumerate(classified_sentences):
    logger.info(f"Classification {i}: {sentence.__dict__}")
    print("---")

2026-05-10 21:17:11,140 - missing_citations - INFO - Classifying sentences for citation worthiness using Gemini Classifier...
2026-05-10 21:17:12,226 - missing_citations - INFO - Sending batch 1/2 to Gemini...
2026-05-10 21:17:18,128 - missing_citations - INFO - Waiting 50 seconds before next Gemini API call...
2026-05-10 21:18:08,130 - missing_citations - INFO - Sending batch 2/2 to Gemini...
2026-05-10 21:18:13,654 - missing_citations - INFO - Classification results:
2026-05-10 21:18:13,655 - missing_citations - INFO - Classification 0: {'text': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers.', 'section': 'Abstract', 'position_in_section': 0.0, 'has_citation': False, 'citation_intent': <CitationIntent.METHOD: 'METHOD'>, 'retrieval_text': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers.', 'previous_sentence': N

In [10]:
"""
STAGE 4B - Urgency Scorer
"""


# 1. Connect to Qdrant Database
client = create_qdrant_client(config.QDRANT_URL)

# 2. Load the Embedding Models (this might take a moment if they aren't downloaded)
dense_model = SentenceTransformer(config.DENSE_MODEL)
sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)

# 3. Initialize the Hybrid Retriever
retriever = HybridRetriever(
    qdrant_client=client,
    dense_model=dense_model,
    sparse_model=sparse_model,
    collection=config.QDRANT_COLLECTION_NAME,
    prefetch_limit=40, # Number of candidates fetched before RRF fusion
)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2027.60it/s]


In [ ]:
scorer = UrgencyScorer(retriever=retriever)
scored_sentences, features = scorer.score_sentences(classified_sentences[:30], 2020)

In [ ]:
for i, s in enumerate(scored_sentences[:30]):
    print(s)

print(features)

SentenceRecord(text='We introduce a new language representation model called BERT...', section='Abstract', pos=0.00, has_cite=False, citation_intent=METHOD, citation_state=NOT_CITATION_WORTHY, worthiness_score=None, urgency_score=None)
SentenceRecord(text='Unlike recent language representation models (Peters et al.,...', section='Abstract', pos=0.25, has_cite=True, citation_intent=BACKGROUND, citation_state=HAS_CITATION, worthiness_score=None, urgency_score=None)
SentenceRecord(text='As a result, the pre-trained BERT model can be fine-tuned wi...', section='Abstract', pos=0.50, has_cite=False, citation_intent=METHOD, citation_state=NOT_CITATION_WORTHY, worthiness_score=None, urgency_score=None)
SentenceRecord(text='BERT is conceptually simple and empirically powerful.', section='Abstract', pos=0.75, has_cite=False, citation_intent=RESULT, citation_state=NOT_CITATION_WORTHY, worthiness_score=None, urgency_score=None)
SentenceRecord(text='It obtains new state-of-the-art results on eleven

In [13]:
# STAGE 5 - Claim Decomposition

claim_decomposer = ClaimDecomposer(model=config.CLASSIFIER_BACKUP[1])
cnt = 0

decomposed_claims: list[Decomposition] = []

for sentence in scored_sentences[:60]:
    if cnt == 5:
        break

    if sentence.urgency_score and sentence.urgency_score >= 0.5:  # Arbitrary threshold for demonstration
        cnt += 1
        decomposed_claims.append(claim_decomposer.decompose(sentence.text))
        logger.info(f"Decomposed claims for sentence: '{sentence.text}'")
        logger.info(f"Decomposed claims: {decomposed_claims[-1]}")

2026-05-10 21:18:33,816 - missing_citations - INFO - Decomposed claims for sentence: 'Language model pre-training has been shown to be effective for improving many natural language processing tasks.'
2026-05-10 21:18:33,817 - missing_citations - INFO - Decomposed claims: Decomposition(
  original_text='Language model pre-training has been shown to be effective for improving many natural language processing tasks.',
  aggregation='AggregationStrategy.WEIGHTED',
  subclaims=[
- Language model pre-training is effective for improving natural language processing tasks. (importance: 1.0)
  ]
)
2026-05-10 21:18:35,104 - missing_citations - INFO - Decomposed claims for sentence: 'There are two existing strategies for applying pre-trained language representations to downstream tasks: feature-based and fine-tuning.'
2026-05-10 21:18:35,105 - missing_citations - INFO - Decomposed claims: Decomposition(
  original_text='There are two existing strategies for applying pre-trained language representa

In [14]:
# Stage 6 — Hybrid Retrieval

from pipeline.aggregator import DecomposedRetriever
from pipeline.reranker import CrossEncoderReranker

reranker = CrossEncoderReranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
decomposed_retriever = DecomposedRetriever(
    retriever=retriever,
    reranker=reranker,
    candidates_per_subclaim=500,
)

for decomposition in decomposed_claims:
    ranked_papers = decomposed_retriever.retrieve_ranked(decomposition, top_k=10, max_year=2018)
    logger.info(f"Results for: '{decomposition.original_text}'")
    for paper in ranked_papers:
        logger.info(
            f"  agg={paper.aggregate_score:.4f}  rerank={paper.result.score:.4f}  "
            f"id={paper.result.paper_id}  title={paper.result.title}"
        )


2026-05-10 21:18:47,971 - missing_citations - INFO - Results for: 'Language model pre-training has been shown to be effective for improving many natural language processing tasks.'
2026-05-10 21:18:47,972 - missing_citations - INFO -   score=0.0164  id=W2898700502
2026-05-10 21:18:47,974 - missing_citations - INFO -   score=0.0161  id=W932413789
2026-05-10 21:18:47,975 - missing_citations - INFO -   score=0.0159  id=W1965154800
2026-05-10 21:18:47,976 - missing_citations - INFO -   score=0.0156  id=W2041145449
2026-05-10 21:18:47,977 - missing_citations - INFO -   score=0.0154  id=W2176263492
2026-05-10 21:18:47,978 - missing_citations - INFO -   score=0.0152  id=W2963248296
2026-05-10 21:18:47,978 - missing_citations - INFO -   score=0.0149  id=W2555428947
2026-05-10 21:18:47,979 - missing_citations - INFO -   score=0.0147  id=W2161222299
2026-05-10 21:18:47,979 - missing_citations - INFO -   score=0.0145  id=W2740711318
2026-05-10 21:18:47,980 - missing_citations - INFO -   score=0.0